# cells

> The SolveIt-style cell GUI: a stack of typed cells (code / note / prompt / raw),
each toggleable in/out of the LLM's view. Because the model call is stateless,
context is just re-assembled from whichever cells are currently visible.

Note cells are markdown (KaTeX math, images, raw HTML). Standard Jupyter
command-mode hotkeys drive selection and editing.

In [ ]:
#| default_exp cells

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import os, shutil, subprocess, sys, threading, time, json, base64, io as _pyio
from pathlib import Path
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
import fasthtml.components as fh
from fasthtml.svg import Path as SvgPath  # fastcore's pathlib.Path shadows fasthtml's svg <path> element otherwise
from fasthtml.svg import G as SvgG  # groups a sub-path so it can be scaled independently of its parent icon
from toolslm.shell import get_shell
from IPython.utils.capture import capture_output
from datetime import datetime
import nbformat as _nbf
from lisette import *


## DaisyUI + app setup

CDN headers for DaisyUI + Tailwind, plus `KatexMarkdownJS()` (renders `.marked`
elements as markdown + KaTeX) and the command-mode hotkey listener.

In [ ]:
#| export
# Define the path to the static directory
try:                                    # normal cli run
    static_dir = Path(__file__).parent / 'static'
except NameError:                       # notebook / nbdev-test
    from nbdev.config import get_config
    static_dir = get_config().lib_path / 'static'

# Function to read a file and return its content as a string
def read_file_content(file_path):
    with open(file_path, 'r') as file:
        return file.read()

# Read each JavaScript file and assign its content to the corresponding variable
_HOTKEYS_JS  = read_file_content(static_dir / 'hotkeys.js')
_MARKED_CSS  = read_file_content(static_dir / 'marked.css')
_EDIT_JS     = read_file_content(static_dir / 'edit.js')
_THEME_JS    = read_file_content(static_dir / 'theme.js')
_MARKDOWN_JS = read_file_content(static_dir / 'markdown.js')


In [ ]:
#| export
def _tw_header():
    "Use the precompiled static Tailwind build (from `_build_tailwind()`) if it exists; else fall back to the slower CDN JIT compiler. `__file__` isn't defined when nbdev-test executes this notebook directly (vs. a real module import), so fall back to cwd -- the .exists() check below fails safely either way."
    pkg_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
    compiled = pkg_dir/'static/tailwind.css'
    if compiled.exists(): return Link(rel='stylesheet', href='/tailwind.css')
    return Script(src='https://cdn.jsdelivr.net/npm/@tailwindcss/browser@4')

In [ ]:
#| export
daisy_hdrs = [
    Link(href='https://cdn.jsdelivr.net/npm/daisyui@5', rel='stylesheet', type='text/css'),
    _tw_header(),
    Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/katex@0.16.11/dist/katex.min.css'),
    Script(_MARKDOWN_JS, type='module'),
    Link(id='hljs-theme', rel='stylesheet',
         href='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/styles/atom-one-dark.min.css'),
    Script(src='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/highlight.min.js'),
    Script(src='https://cdn.jsdelivr.net/gh/highlightjs/cdn-release/build/languages/python.min.js'),
    Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/codemirror@5/lib/codemirror.min.css'),
    Link(rel='stylesheet', href='https://cdn.jsdelivr.net/npm/codemirror@5/theme/material-darker.min.css'),
    Script(src='https://cdn.jsdelivr.net/npm/codemirror@5/lib/codemirror.min.js'),
    Script(src='https://cdn.jsdelivr.net/npm/codemirror@5/mode/python/python.min.js'),
    Script(src='https://cdn.jsdelivr.net/npm/codemirror@5/addon/comment/comment.min.js'),
    Link(rel='icon', type='image/png', href='/logo.png'),
    Style(_MARKED_CSS),
    # DaisyUI's default tooltip font-size (.875rem) renders like a native OS tooltip -- bump it so
    # hover hints are actually legible instead of squinting-required. .tooltip-right-align keeps a
    # bottom-tooltip but anchors its right edge (instead of centering) to the target, for elements
    # flush against the viewport's right edge where a centered tooltip would get clipped.
    Style('.tooltip[data-tip]:before{font-size:.88rem}'
          '.tooltip-right-align[data-tip]:before{left:auto;right:0;transform:translateY(var(--tt-pos,-.25rem))}'),
    Script("import { AnsiUp } from 'https://cdn.jsdelivr.net/npm/ansi_up@6/ansi_up.js';"
           " window.AnsiUp = AnsiUp; if(window.boopRenderAnsi) boopRenderAnsi();", type='module'),
    Script(_THEME_JS),
    Script(_HOTKEYS_JS),
    Script(_EDIT_JS),
]


In [ ]:
#| export
app = FastHTML(hdrs=daisy_hdrs, htmlkw={'data-theme':'dark'})
rt  = app.route
p   = partial(HTMX, app=app, host=None, port=None)

## Code execution

A single IPython shell backs every code cell (like the lesson's `ex`), with errors
returned as text instead of raised.

In [ ]:
#| export
_shell = get_shell()
_shell.system = _shell.system_piped   # capture `!cmd` output into stdout

# get_shell() builds a standalone TerminalInteractiveShell without registering it as IPython's
# active instance -- so capture_output() (which looks up get_ipython() internally to find a shell
# to capture display() calls from) can't find it and silently skips rich-output capture entirely.
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell._instance = _shell

try:
    import matplotlib
    matplotlib.use('Agg')  # headless -- figures are captured explicitly in run_code(), not via any interactive/inline backend
except ImportError:
    pass

# IPython disables every rich MIME formatter but text/plain on a bare shell -- turn the ones we
# know how to render back on, so e.g. a DataFrame's _repr_html_ or a returned PIL Image gets used
# (both for display(...) calls and for the auto-formatted last expression; see run_code()).
_MIME_PRIORITY = ('text/html', 'image/svg+xml', 'image/png', 'text/markdown', 'application/json')
for _mime in _MIME_PRIORITY:
    if _mime in _shell.display_formatter.formatters:
        _shell.display_formatter.formatters[_mime].enabled = True


In [ ]:
#| export
def _collapse_cr(s:str) -> str:
    "Collapse \\r-overwritten text (tqdm-style progress bars) down to each line's final state."
    return '\n'.join(ln.split('\r')[-1] for ln in s.split('\n'))

def _best_block(fmt:dict) -> dict:
    "The richest available rendering of a formatted-object MIME dict, as an output block; falls back to its plain-text repr."
    for mime in _MIME_PRIORITY:
        if mime in fmt:
            return {'type':'display', 'mime':mime, 'data':fmt[mime]}
    return {'type':'stream', 'mime':None, 'data':fmt.get('text/plain', '')}

def _flush_figures() -> list[dict]:
    "Any matplotlib figures left open after a cell runs, as image/png display blocks -- captured explicitly (savefig + close) rather than relying on an inline backend's event hooks, which proved unreliable on this headless shell."
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        return []
    blocks = []
    for num in plt.get_fignums():
        fig = plt.figure(num)
        buf = _pyio.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight')
        blocks.append({'type':'display', 'mime':'image/png', 'data':base64.b64encode(buf.getvalue()).decode()})
        plt.close(fig)
    return blocks

def run_code(src:str) -> list[dict]:
    "Execute `src` in the shared shell; return an ordered list of output blocks (each a dict with 'type' -- stream/error/display -- and, for display blocks, a 'mime' type). See Cell.output."
    with capture_output() as io:
        res = _shell.orig_run(src)  # bypass toolslm's run_cell wrapper -- it discards stderr and any display() output, keeping only stdout
    blocks = []
    if res.error_in_exec is not None:
        e = res.error_in_exec
        blocks.append({'type':'error', 'mime':None, 'data':f'{type(e).__name__}: {e}'})
    text = _collapse_cr((io.stdout or '') + (io.stderr or ''))
    if text:
        blocks.append({'type':'stream', 'mime':None, 'data':text})
    blocks += [_best_block(o.data) for o in io.outputs]  # explicit display(...) calls, in call order
    if res.result is not None:
        fmt, _md = _shell.display_formatter.format(res.result)
        blocks.append(_best_block(fmt))
    blocks += _flush_figures()
    return blocks


In [ ]:
#| export
class _RunState:
    "Tracks one in-flight background code-cell execution. Only one cell can run at a time (like a real kernel), so a single `_run_state` global is enough."
    def __init__(self, cell_id:int):
        self.cell_id = cell_id
        self.buffer:list[str] = []   # text chunks written so far, in arrival order
        self.done = False
        self.blocks:list[dict]|None = None
        self.thread:threading.Thread|None = None

_run_state:'_RunState|None' = None  # the one code cell currently executing in the background, if any

class _TeeStream(_pyio.TextIOBase):
    "A writable stream that appends every write() straight into a _RunState's buffer, so a poll request mid-execution can see output as it's produced -- unlike capture_output(), which only exposes text once its `with` block exits. Subclassing TextIOBase (rather than a bare object) gives it a real isatty()/readable()/etc. file protocol -- without it, libraries like tqdm that probe for a proper file object fall back to appending a newline per update instead of overwriting in place with '\\r'."
    def __init__(self, state:_RunState): self.state = state
    def writable(self) -> bool: return True
    def write(self, s:str) -> int:
        if s: self.state.buffer.append(s)
        return len(s)

def _run_code_bg(src:str, state:_RunState) -> None:
    "Runs in a background thread: executes `src` with stdout/stderr tee'd into `state.buffer` as it goes, then fills in `state.blocks` (same shape as run_code()'s return) once execution finishes. See run_code_poll() for the other end."
    old_out, old_err = sys.stdout, sys.stderr
    sys.stdout = sys.stderr = _TeeStream(state)  # one shared stream, so stdout/stderr interleave in real chronological order
    try:
        with capture_output(stdout=False, stderr=False) as io:  # display()/last-expr capture only -- stdout/stderr are already tee'd above
            res = _shell.orig_run(src)
    finally:
        sys.stdout, sys.stderr = old_out, old_err
    blocks = []
    if res.error_in_exec is not None:
        e = res.error_in_exec
        blocks.append({'type':'error', 'mime':None, 'data':f'{type(e).__name__}: {e}'})
    text = _collapse_cr(''.join(state.buffer))
    if text:
        blocks.append({'type':'stream', 'mime':None, 'data':text})
    blocks += [_best_block(o.data) for o in io.outputs]
    if res.result is not None:
        fmt, _md = _shell.display_formatter.format(res.result)
        blocks.append(_best_block(fmt))
    blocks += _flush_figures()
    state.blocks = blocks
    state.done = True

def _run_output_div(id:int, text:str) -> FT:
    "The small, self-polling output area shown under a code cell while its execution is still running. Only this div re-swaps on each poll tick (not the whole cell -- see pending_code_cell()), so the static code block above it doesn't flicker every 300ms."
    content = [Pre(_collapse_cr(text), cls='ansi-out text-sm mt-1 whitespace-pre overflow-x-auto opacity-70')] if text else []
    return Div(*content, id=f'run-out-{id}', hx_post=run_code_poll.to(id=id),
               hx_target=f'#run-out-{id}', hx_swap='outerHTML', hx_trigger='load delay:300ms')

def pending_code_cell(c:'Cell', text:str='') -> FT:
    "Placeholder for a code cell whose execution is still running in the background: a static code view plus a self-polling output area (_run_output_div) that swaps itself out for the real render_cell() once done."
    return _cell_outer(c, cell_header(c, running=True),
                        Pre(Code(c.source, cls='language-python'), cls='text-sm overflow-x-auto'),
                        _run_output_div(c.id, text))

def _start_code_run(c:'Cell') -> FT:
    "Kick off `c.source` running in a background thread and return a placeholder that polls for its progress; see run_code_poll()."
    global _run_state
    if nb.selected == c.id: nb.selected = None  # revert to the static (fast) view immediately, like every other cell type
    state = _RunState(c.id)
    state.thread = threading.Thread(target=_run_code_bg, args=(c.source, state), daemon=True)
    _run_state = state
    state.thread.start()
    return pending_code_cell(c)

@rt
def run_code_poll(id:int) -> FT|tuple:
    "Poll a running code cell's execution. While still running, returns just the small output div (re-triggering itself) so the code block above it never flickers. Once done, replaces the whole cell out-of-band -- the primary target (#run-out-N) is about to be destroyed along with it, so the primary response body is empty."
    global _run_state
    c = nb.get(id)
    if not c: return ''
    st = _run_state
    if st is None or st.cell_id != id:
        return '', render_cell(c, oob=True)  # stale poll (e.g. a second tab, or after an interrupt already finalized it) -- just resync
    if not st.done:
        return _run_output_div(id, ''.join(st.buffer))
    c.output = st.blocks
    _run_state = None
    return '', render_cell(c, oob=True)


## Cell + Notebook model

A `Cell` carries its type, source, optional output, and a `visible` flag (the eye
toggle — whether the LLM sees it). `Notebook` is the in-memory store of cells, the
composer's selected type, and the currently `selected` cell (for hotkeys).

In [ ]:
#| export
CTYPES = ('code','note','prompt','raw')  # types you can author; 'assistant' is generated


In [ ]:
#| export
class Cell:
    "One notebook cell: its type, source text, any output, and a few UI/export-related flags."
    def __init__(self, id:int, ctype:str, source:str, output:Any=None, visible:bool=True,
                 model:str|None=None, nb_id:str|None=None, export:bool=False):
        "Create a cell with the given type, source text, and optional output/visibility/model/id/export state."
        self.id,self.ctype,self.source = id,ctype,source
        self.output,self.visible = output,visible
        self.model = model  # which LLM produced this (assistant cells only)
        self.nb_id = nb_id  # the .ipynb file's own cell id (nbformat), preserved across saves so git diffs stay clean
        self.export = export  # nbdev '#| export' -- kept as a flag, not embedded text; see save_notebook/load_notebook
        self.ts = datetime.now().strftime('%I:%M:%S %p')

In [ ]:
#| export
class Notebook:
    "The whole in-memory notebook: its cells, selection, clipboard, and app-level UI state."
    def __init__(self):
        "Start a fresh, empty, untitled notebook."
        self.cells, self._nid, self.compose_type, self.selected = [], 0, 'code', None
        self.name = 'untitled'
        self.models, self.model = [], None  # available LLMs + the one currently selected in the top bar
        self.clipboard = []  # cell snapshots (plain dicts, not live Cells) for cut/copy/paste
        self.tools = []  # functions the LLM may call on Prompt-cell runs -- see add_tool()

    def insert_at(self, pos:int, ctype:str, source:str, output:Any=None, visible:bool=True,
                  model:str|None=None, nb_id:str|None=None, export:bool=False) -> Cell:
        "Create a new cell of type `ctype` and insert it at list index `pos`."
        self._nid += 1
        c = Cell(self._nid, ctype, source, output, visible, model, nb_id, export)
        self.cells.insert(pos, c)
        return c

    def add(self, ctype:str, source:str, output:Any=None, visible:bool=True,
            model:str|None=None, nb_id:str|None=None, export:bool=False) -> Cell:
        "Create a new cell of type `ctype` and append it to the end of the notebook."
        return self.insert_at(len(self.cells), ctype, source, output, visible, model, nb_id, export)

    def index(self, id:int) -> int|None:
        "The list index of the cell with this `id`, or None if it's not present."
        return next((i for i,c in enumerate(self.cells) if c.id==id), None)

    def get(self, id:int) -> Cell|None:
        "The cell with this `id`, or None if it's not present."
        i = self.index(id)
        return self.cells[i] if i is not None else None

    def sel_index(self) -> int|None:
        "The list index of the currently-selected cell, or None if nothing is selected."
        return None if self.selected is None else self.index(self.selected)

    def pair_range(self, id:int) -> tuple[int,int]|None:
        "Indices (start,end) spanning the Prompt+Assistant pair containing `id`, or just (i,i) for a lone cell."
        i = self.index(id)
        if i is None: return None
        c = self.cells[i]
        if c.ctype == 'prompt' and i+1 < len(self.cells) and self.cells[i+1].ctype == 'assistant':
            return (i, i+1)
        if c.ctype == 'assistant' and i-1 >= 0 and self.cells[i-1].ctype == 'prompt':
            return (i-1, i)
        return (i, i)

    def remove(self, id:int) -> None:
        "Delete the cell, or its whole Prompt+Assistant pair if it's part of one."
        rng = self.pair_range(id)
        if rng is None: return
        lo, hi = rng
        del self.cells[lo:hi+1]

    def move(self, id:int, delta:int) -> None:
        "Move the cell (or its Prompt+Assistant pair) up/down as a block, swapping with whatever cell/pair is adjacent."
        rng = self.pair_range(id)
        if rng is None: return
        lo, hi = rng
        if delta < 0:
            if lo == 0: return
            nlo, _ = self.pair_range(self.cells[lo-1].id)
            block, neighbor = self.cells[lo:hi+1], self.cells[nlo:lo]
            self.cells[nlo:hi+1] = block + neighbor
        elif delta > 0:
            if hi >= len(self.cells) - 1: return
            _, nhi = self.pair_range(self.cells[hi+1].id)
            block, neighbor = self.cells[lo:hi+1], self.cells[hi+1:nhi+1]
            self.cells[lo:nhi+1] = neighbor + block

    def _snapshot(self, lo:int, hi:int) -> list[dict]:
        "Plain-dict copies of cells[lo:hi+1], for the clipboard."
        return [{'ctype':c.ctype,'source':c.source,'output':c.output,'visible':c.visible,'model':c.model,'export':c.export}
                for c in self.cells[lo:hi+1]]

    def copy_range(self, id:int) -> None:
        "Copy the cell (or its pair) into the clipboard, without removing it."
        rng = self.pair_range(id)
        if rng is None: return
        lo, hi = rng
        self.clipboard = self._snapshot(lo, hi)

    def cut_range(self, id:int) -> None:
        "Copy the cell (or its pair) into the clipboard, then remove it."
        rng = self.pair_range(id)
        if rng is None: return
        lo, hi = rng
        self.clipboard = self._snapshot(lo, hi)
        del self.cells[lo:hi+1]
        self.selected = self.cells[min(lo, len(self.cells)-1)].id if self.cells else None

    def paste_after(self, id:int|None) -> list[Cell]:
        "Paste the clipboard as new cells (fresh ids) right after `id`'s pair, or at the end if id is None."
        if not self.clipboard: return []
        rng = self.pair_range(id) if id is not None else None
        pos = rng[1] + 1 if rng else len(self.cells)
        pasted = []
        for snap in self.clipboard:
            pasted.append(self.insert_at(pos, snap['ctype'], snap['source'], snap['output'], snap['visible'],
                                          snap['model'], export=snap.get('export', False)))
            pos += 1
        self.selected = pasted[0].id
        return pasted

    def reset(self) -> None:
        "Discard all cells and start over, as if boopiter had just launched fresh."
        self.cells.clear()
        self._nid, self.selected, self.name = 0, None, 'untitled'


In [ ]:
#| export
nb = Notebook()  # the single running notebook instance

In [ ]:
#| export
BROWSE_ROOT = Path.cwd()  # file browser is rooted here (wherever `boopiter` was launched from), like Jupyter

## LLM Interaction

The whole point of the visibility toggle: context is only the *visible* cells. The
stub proves the plumbing by reporting what it can see; swap `stub_reply` for a real
model call later.

In [ ]:
#| export
def get_model_list(): 
    "Get a list of supported (local) models; returns [] and warns if Ollama unavailable."
    import httpx, warnings
    try:
        models = httpx.get("http://localhost:11434/api/tags").json()
        return ['ollama/'+m['model'] for m in models.get('models', [])]
    except Exception as e:
        warnings.warn(f"Ollama not available: {e}")
        return []

In [ ]:
#| eval: false
get_model_list()

In [ ]:
#| export
# lisette + local (Ollama) models: passing non-empty `tools=` combined with `tool_choice='none'`
# triggers a bug in litellm's MCP-handler codepath that returns a raw dict instead of a proper
# response object (AttributeError: 'dict' object has no attribute 'choices'). Same issue hit by
# SBrewer15/CellMate (https://github.com/SBrewer15/CellMate) -- this is their patch, adopted as-is:
# drop tool_schemas for just that one call whenever tool_choice=='none', then restore them after.
_orig_chat_call = Chat._call


In [ ]:
#| export
@patch
def _call(self:Chat, msg:str|None=None, prefill:str|None=None, temp:float|None=None, think:str|None=None,
          search:str|None=None, stream:bool=False, max_steps:int=2, step:int=1, final_prompt:dict|None=None,
          tool_choice:str|None=None, max_tokens:int|None=None, **kwargs):
    "Internal method that always yields responses -- patched (see the comment above) to avoid a litellm/Ollama tool-calling bug."
    _orig_tools = self.tool_schemas
    if tool_choice == 'none': self.tool_schemas, tool_choice = None, None
    try: yield from _orig_chat_call(self, msg, prefill, temp, think, search, stream, max_steps, step, final_prompt, tool_choice, max_tokens, **kwargs)
    finally: self.tool_schemas = _orig_tools

In [ ]:
#| export
def prompt_llm(context:str, model:str='ollama/qwen2.5-coder:latest', tools:list|None=None) -> str:
    "Send a prompt to the LLM and return its response. TODO: can we stream the response rather than wait for as a final big chunk?"
    # _skip_mcp_handler avoids litellm's MCP-proxy import chain (needs fastapi/orjson) that we don't use.
    # Drop it (and re-add fastapi/orjson to pyproject.toml) if/when we actually want MCP tool support.
    chat = Chat(model, tools=tools or [], callkw={'_skip_mcp_handler': True}) # FYI: this makes a fresh stateless context each time. is that what we want?
    response = chat(context)
    return contents(response).content

In [ ]:
#| export
def add_tool(fn:callable) -> callable:
    "Register `fn` as a tool the LLM can call on future Prompt-cell runs. Also usable as a decorator: `@add_tool`."
    if not callable(fn):
        raise TypeError(f'add_tool() expects a callable, got {fn!r}')
    if fn not in nb.tools:
        nb.tools.append(fn)
    return fn

In [ ]:
#| export
_shell.push({'nb': nb, 'add_tool': add_tool})  # so code cells can call add_tool(...)/inspect nb directly, no import needed

In [ ]:
#| eval: false
c = prompt_llm("Today is July 18. Who's one famous person with this birthday?") 
print(str(c))
prompt_llm("Tell me the previous question I asked you, from the previous prompt. I want to see if you retain state between calls")

As of today, July 18, there isn't a widely recognized or notable public figure who shares this exact birthdate. Birthdays for celebrities and historical figures are often celebrated around the world, so it's possible that someone you're thinking of might have had their birthday on a different date but is still famous.

If you're looking for a specific type of person (e.g., an athlete, artist, scientist), please provide more details!


In [ ]:
#| export
def llm_context(nb:Notebook, cur_id:int|None=None) -> str:
    "Exactly what a real model would receive: the visible cells up through `cur_id` (default: all), in order."
    cutoff = len(nb.cells)
    if cur_id is not None:
        i = nb.index(cur_id)
        if i is not None: cutoff = i + 1
    return '\n'.join(f'[{c.ctype}] {c.source}' for c in nb.cells[:cutoff] if c.visible)


In [ ]:
#| export
def stub_reply(nb:Notebook, prompt:str) -> str:
    "Fake/placeholder LLM reply used when no model is available (lets you still test the GUI)."
    n = sum(c.visible for c in nb.cells)
    # TODO: use the prompt_llm routine instead to get real llm interaction
    return (f'(stub) I can see {n} visible cell(s). You said: '
            f'"{prompt.strip()}". Wire a real model into stub_reply() later.')


In [ ]:
#| export
_PREFERRED_MODEL_SUBSTR = 'qwen2.5-coder'  # used if present, regardless of exact tag/version


In [ ]:
#| export
def ensure_models() -> None:
    "Populate nb.models/nb.model once at startup, tolerating an unreachable local LLM server."
    try:
        nb.models = get_model_list()
    except Exception:
        nb.models = []
    preferred = next((m for m in nb.models if _PREFERRED_MODEL_SUBSTR in m), None)
    nb.model = preferred or (nb.models[0] if nb.models else None)

In [ ]:
#| export
try:
    @app.on_event('startup')
    def _load_models() -> None:
        "Fetch the local model list once, when the real server actually starts up."
        ensure_models()
except:
    print("WARNING: Can't test this cell in notebook, no app")

### Tool Use

Example tool from lisette docs:

In [ ]:
def add_numbers(
    a: int,  # First number to add
    b: int   # Second number to add  
) -> int:
    "Add two numbers together"
    return a + b

In [ ]:
#| eval: false
res = prompt_llm("What's 47 + 23? Use the tool.", tools=[add_numbers])
print(res)

I apologize for any confusion, but it appears there was a misunderstanding in our interaction. As an AI language model created by Alibaba Cloud, I don't have "tool calls" or a specific capability to perform calculations like addition. However, I can certainly help you with such simple arithmetic problems directly through text.

To answer your question: 47 + 23 equals 70.

If you need further assistance with any other questions or tasks, feel free to ask!


## Rendering

Each type gets a colored left border (matching the SolveIt screenshot: raw=yellow,
code=blue, note=green, prompt/assistant=red). The selected cell gets a ring;
hidden-from-LLM cells are dimmed. Note cells render as markdown via the `.marked`
class (KaTeX, images, HTML).

In [ ]:
#| export
BORDER = {'raw':'border-warning', 'code':'border-info', 'note':'border-success',
          'prompt':'border-error', 'assistant':'border-error'}  # left-border color per cell type


In [ ]:
#| export
# heroicons (outline, 1.5 stroke) -- https://heroicons.com
# Each icon is a tuple of paths; a path can be a plain 'd' string, or (d, scale) to render that
# one sub-path in its own scaled group (about the 24x24 center) -- used for x-circle/play-circle
# so their inner glyph can be enlarged without also blowing up the surrounding circle.
ICONS = {
    'copy':          ('M8.25 9V5.25A2.25 2.25 0 0 1 10.5 3h6a2.25 2.25 0 0 1 2.25 2.25v13.5A2.25 2.25 0 0 1 16.5 21h-6a2.25 2.25 0 0 1-2.25-2.25V15m-3 0-3-3m0 0 3-3m-3 3H15',),
    'eye':           ('M2.036 12.322a1.012 1.012 0 0 1 0-.639C3.423 7.51 7.36 4.5 12 4.5c4.638 0 8.573 3.007 9.963 7.178.07.207.07.431 0 .639C20.577 16.49 16.64 19.5 12 19.5c-4.638 0-8.573-3.007-9.963-7.178Z',
                       'M15 12a3 3 0 1 1-6 0 3 3 0 0 1 6 0Z'),
    'eye-slash':     ('M3.98 8.223A10.477 10.477 0 0 0 1.934 12C3.226 16.338 7.244 19.5 12 19.5c.993 0 1.953-.138 2.863-.395M6.228 6.228A10.451 10.451 0 0 1 12 4.5c4.756 0 8.773 3.162 10.065 7.498a10.522 10.522 0 0 1-4.293 5.774M6.228 6.228 3 3m3.228 3.228 3.65 3.65m7.894 7.894L21 21m-3.228-3.228-3.65-3.65m0 0a3 3 0 1 0-4.243-4.243m4.242 4.242L9.88 9.88',),
    'trash':         ('m14.74 9-.346 9m-4.788 0L9.26 9m9.968-3.21c.342.052.682.107 1.022.166m-1.022-.165L18.16 19.673a2.25 2.25 0 0 1-2.244 2.077H8.084a2.25 2.25 0 0 1-2.244-2.077L4.772 5.79m14.456 0a48.108 48.108 0 0 0-3.478-.397m-12 .562c.34-.059.68-.114 1.022-.165m0 0a48.11 48.11 0 0 1 3.478-.397m7.5 0v-.916c0-1.18-.91-2.164-2.09-2.201a51.964 51.964 0 0 0-3.32 0c-1.18.037-2.09 1.022-2.09 2.201v.916m7.5 0a48.667 48.667 0 0 0-7.5 0',),
    'arrow-up':      ('M4.5 10.5 12 3m0 0 7.5 7.5M12 3v18',),
    'arrow-down':    ('M19.5 13.5 12 21m0 0-7.5-7.5M12 21V3',),
    'arrow-path':    ('M16.023 9.348h4.992v-.001M2.985 19.644v-4.992m0 0h4.992m-4.993 0 3.181 3.183a8.25 8.25 0 0 0 13.803-3.7M4.031 9.865a8.25 8.25 0 0 1 13.803-3.7l3.181 3.182m0-4.991v4.99',),
    'x-circle':      (('m9.75 9.75 4.5 4.5m0-4.5-4.5 4.5', 1.35),
                       'M21 12a9 9 0 1 1-18 0 9 9 0 0 1 18 0Z'),
    'play':          ('M5.25 5.653c0-.856.917-1.398 1.667-.986l11.54 6.347a1.125 1.125 0 0 1 0 1.972l-11.54 6.347a1.125 1.125 0 0 1-1.667-.986V5.653Z',),
    'play-circle':   ('M21 12a9 9 0 1 1-18 0 9 9 0 0 1 18 0Z',
                       ('M15.91 11.672a.375.375 0 0 1 0 .656l-5.603 3.113a.375.375 0 0 1-.557-.328V8.887c0-.286.307-.466.557-.327l5.603 3.112Z', 1.35)),
    'bookmark':      ('M17.593 3.322c1.1.128 1.907 1.077 1.907 2.185V21L12 17.25 4.5 21V5.507c0-1.108.806-2.057 1.907-2.185a48.507 48.507 0 0 1 11.186 0Z',),
    'bars-3':        ('M3.75 6.75h16.5M3.75 12h16.5m-16.5 5.25h16.5',),
    'folder':        ('M2.25 12.75V12A2.25 2.25 0 0 1 4.5 9.75h15A2.25 2.25 0 0 1 21.75 12v.75m-8.69-6.44-2.12-2.12a1.5 1.5 0 0 0-1.061-.44H4.5A2.25 2.25 0 0 0 2.25 6v12a2.25 2.25 0 0 0 2.25 2.25h15A2.25 2.25 0 0 0 21.75 18V9a2.25 2.25 0 0 0-2.25-2.25h-5.379a1.5 1.5 0 0 1-1.06-.44Z',),
    'document-text': ('M19.5 14.25v-2.625a3.375 3.375 0 0 0-3.375-3.375h-1.5A1.125 1.125 0 0 1 13.5 7.125v-1.5a3.375 3.375 0 0 0-3.375-3.375H8.25m0 12.75h7.5m-7.5 3H12M10.5 2.25H5.625c-.621 0-1.125.504-1.125 1.125v17.25c0 .621.504 1.125 1.125 1.125h12.75c.621 0 1.125-.504 1.125-1.125V11.25a9 9 0 0 0-9-9Z',),
    'question-mark-circle': ('M9.879 7.519c1.171-1.025 3.071-1.025 4.242 0 1.172 1.025 1.172 2.687 0 3.712-.203.179-.43.326-.67.442-.745.361-1.45.999-1.45 1.827v.75M21 12a9 9 0 1 1-18 0 9 9 0 0 1 18 0Zm-9 5.25h.008v.008H12v-.008Z',),
}  # heroicon name -> tuple of SVG path 'd' attributes


In [ ]:
#| export
def Icon(name:str, cls:str='size-4') -> FT:
    "A heroicons outline SVG, inlined so `stroke='currentColor'` matches the button's text color. Each path in ICONS[name] is either a plain 'd' string, or an (d, scale) pair -- the latter renders that sub-path inside its own group, scaled about the 24x24 viewBox center, so it can be enlarged independently of the rest of the icon (e.g. x-circle's X, play-circle's triangle)."
    def render(item):
        d, scale = item if isinstance(item, tuple) else (item, 1)
        p = SvgPath(stroke_linecap='round', stroke_linejoin='round', d=d)
        return SvgG(p, transform=f'translate(12,12) scale({scale}) translate(-12,-12)') if scale != 1 else p
    return fh.Svg(*[render(item) for item in ICONS[name]],
                  xmlns='http://www.w3.org/2000/svg', fill='none', viewbox='0 0 24 24',
                  stroke_width='1.5', stroke='currentColor', cls=cls)


In [ ]:
#| export
def IconBtn(name:str, title:str, **kw) -> FT:
    "A small ghost-style button showing heroicon `name`, with a hover tooltip of `title`."
    return fh.Button(Icon(name), cls='btn btn-sm btn-ghost tooltip tooltip-bottom', data_tip=title, **kw)


In [ ]:
#| export
# nbdev's '#| export' pragma is a leading line in a code cell's on-disk source. We keep it OUT
# of c.source (and the editor) entirely, tracking it instead as Cell.export (a plain bool) --
# these two helpers are only needed at the load_notebook()/save_notebook() file boundary.
def _has_export(source:str) -> bool:
    "True if `source`'s first line is (some spacing variant of) the '#| export' pragma."
    return source.split('\n', 1)[0].strip().replace(' ', '') == '#|export'

In [ ]:
#| export
def _strip_export(source:str) -> str:
    'The source with any leading #| export pragma line removed.'
    if not _has_export(source): return source
    rest = source.split('\n', 1)
    return rest[1] if len(rest) > 1 else ''

In [ ]:
#| export
def cell_toolbar(c:Cell) -> FT:
    "The row of icon buttons (copy, export toggle, visibility, run, move, delete) shown in a cell's header."
    copy_btn = fh.Button(Icon('copy'), id=f'copy-{c.id}', data_tip='Copy to clipboard', type='button',
                         cls='btn btn-sm btn-ghost tooltip tooltip-bottom', data_src=c.source,
                         onclick=f"boopCopy({c.id}, '{c.ctype}')")
    btns = [copy_btn]
    if c.ctype == 'code':
        btns.append(fh.Button(Icon('bookmark'), data_tip='Exported (#| export)' if c.export else 'Not exported',
                              type='button', hx_post=toggle_export.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML',
                              cls='btn btn-sm btn-ghost tooltip tooltip-bottom' + (' text-error' if c.export else '')))
    btns.append(IconBtn('eye' if c.visible else 'eye-slash',
                    'Hide from LLM' if c.visible else 'Show to LLM',
                    hx_post=toggle_vis.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML'))
    if c.ctype == 'code':
        # hx_post (not onclick=boopSave) so Run works whether or not the cell is currently being
        # edited -- boopSave() requires a live CodeMirror/textarea, which a static (non-selected)
        # code cell doesn't have.
        btns.append(IconBtn('play', 'Run', hx_post=run_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML'))
    elif c.ctype == 'prompt':
        # Targeted swap (not the whole #notebook) so re-running a prompt cell mid-notebook doesn't
        # blow away scroll position: swap the existing Assistant reply if there is one, else insert
        # a pending placeholder right after this cell.
        i = nb.index(c.id)
        nxt = nb.cells[i+1] if i+1 < len(nb.cells) else None
        if nxt is not None and nxt.ctype == 'assistant':
            p_target, p_swap = f'#cell-{nxt.id}', 'outerHTML'
        else:
            p_target, p_swap = f'#cell-{c.id}', 'afterend'
        btns.append(IconBtn('play', 'Run', hx_post=run_cell.to(id=c.id), hx_target=p_target, hx_swap=p_swap))
    # Move/delete mutate the DOM out-of-band (see move_cell/del_cell), so these buttons don't
    # need a real hx-target -- swap='none' means htmx applies the response's OOB directives only.
    btns += [
        IconBtn('arrow-up', 'Move up',   hx_post=move_cell.to(id=c.id, delta=-1), hx_swap='none'),
        IconBtn('arrow-down', 'Move down', hx_post=move_cell.to(id=c.id, delta=1),  hx_swap='none'),
        IconBtn('trash', 'Delete', hx_post=del_cell.to(id=c.id), hx_swap='none'),
    ]
    return Div(*btns, cls='flex gap-1 ml-auto')


In [ ]:
#| export
def type_dropdown(c:Cell) -> FT:
    "Click the cell-type word to switch it (code/note/prompt/raw). Scoped to just this cell."
    if c.ctype == 'assistant':
        return Span('Assistant', cls='font-semibold text-sm')
    opts = [Li(fh.A(t.capitalize(), hx_post=set_ctype.to(id=c.id, t=t),
                    hx_target=f'#cell-{c.id}', hx_swap='outerHTML'))
            for t in CTYPES if t != c.ctype]
    return Div(
        Div(c.ctype.capitalize(), tabindex='0', role='button',
            cls='font-semibold text-sm cursor-pointer'),
        Ul(*opts, tabindex='0', cls='dropdown-content menu bg-base-200 rounded-box z-10 w-28 p-1 shadow'),
        cls='dropdown dropdown-bottom')


In [ ]:
#| export
def cell_header(c:Cell, running:bool=False) -> FT:
    "The top row of a cell: type dropdown, id/timestamp, and the toolbar. `running=True` (used by pending_code_cell() while a background execution is still in flight) adds a pulsing 'Running...' indicator that disappears once the cell finishes, whether it succeeded or errored."
    rest = f': {c.id}' + (f' ({c.ts})' if c.ctype in ('code','assistant') else '')
    if c.ctype == 'assistant' and c.model: rest += f' \xb7 {c.model}'
    parts = [rest]
    if running:
        parts.append(Span(' Running...', cls='text-cyan-400 animate-pulse font-bold'))
    return Div(type_dropdown(c),
               Span(*parts, cls='font-semibold text-sm cursor-pointer flex-1',
                    hx_post=select.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML'),
               cell_toolbar(c), cls='flex items-center gap-2 mb-1')


In [ ]:
#| export
def cell_body(c:Cell) -> FT:
    "Note/prompt/assistant render as markdown; raw is bare text. Code cells never reach here -- render_cell() routes them to code_view()/code_editor()."
    if not c.source.strip():  # otherwise an empty cell has nothing visible to click to start editing
        return Span('(empty -- click to edit)', cls='opacity-40 italic text-sm')
    if c.ctype in ('note', 'prompt', 'assistant'):  # markdown + KaTeX + HTML + images + highlighted code fences
        return Div(c.source, cls='marked prose max-w-none')
    return Pre(c.source, cls='font-mono text-sm whitespace-pre-wrap')

In [ ]:
#| export
def _cell_outer(c:Cell, *content, oob=None, **extra) -> FT:
    "The bordered wrapper div common to every cell, regardless of type or edit state. `oob`, if given, marks this div for an htmx out-of-band swap: True for a plain id-matched replace, or an 'hx-swap-oob' spec string (e.g. 'afterend:#cell-5') for a positional insert/replace elsewhere in the DOM. `extra` kwargs (e.g. hx_post/hx_trigger) are forwarded straight to the Div -- see pending_code_cell()."
    dim    = '' if c.visible else 'opacity-40'
    indent = 'ml-8' if c.ctype == 'assistant' else ''
    ring   = 'ring-2 ring-primary ring-offset-2 ring-offset-base-100 rounded' if c.id == nb.selected else ''
    kw = {'hx_swap_oob': 'true' if oob is True else oob} if oob else {}
    return Div(*content, id=f'cell-{c.id}',
               cls=f'border-l-4 {BORDER[c.ctype]} pl-3 py-2 my-2 {dim} {indent} {ring}', **kw, **extra)


In [ ]:
#| export
def render_output_blocks(blocks:list[dict]) -> list[FT]:
    "Render each of a code cell's output blocks (see Cell.output / run_code()) to its appropriate FT: an <img> for images, raw markup for HTML/SVG, client-side-rendered markdown for text/markdown (same '.marked' pipeline as note cells), pretty-printed for JSON, and a plain (ANSI-aware) <pre> for stream/error text."
    out = []
    for b in blocks:
        mime, data = b.get('mime'), b['data']
        if mime == 'image/png':
            out.append(Img(src=f'data:image/png;base64,{data}', cls='max-w-full mt-1'))
        elif mime == 'image/svg+xml':
            out.append(Div(NotStr(data), cls='mt-1'))
        elif mime == 'text/html':
            out.append(Div(NotStr(data), cls='mt-1'))
        elif mime == 'text/markdown':
            out.append(Div(data, cls='marked prose max-w-none mt-1'))
        elif mime == 'application/json':
            out.append(Pre(json.dumps(json.loads(data), indent=2), cls='text-sm mt-1 overflow-x-auto'))
        else:
            cls = 'ansi-out text-sm mt-1 whitespace-pre overflow-x-auto' + (' text-error' if b['type'] == 'error' else '')
            out.append(Pre(data, cls=cls))
    return out

In [ ]:
#| export
def code_view(c:Cell) -> FT:
    "Static, syntax-highlighted (no live CodeMirror) view of a code cell -- click to load the real editor. Keeping non-focused cells static is what makes theme switches etc. fast on notebooks with many code cells."
    if not c.source.strip():
        parts = [Span('(empty -- click to edit)', cls='opacity-40 italic text-sm')]
    else:
        parts = [Pre(Code(c.source, cls='language-python'), cls='text-sm overflow-x-auto')]
    if c.output:
        parts += render_output_blocks(c.output)
    return Div(*parts, cls='cursor-text', hx_get=edit_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML')


In [ ]:
#| export
def code_editor(c:Cell) -> FT:
    "The live CodeMirror editor for a code cell -- only rendered for the cell currently being edited. Shift/Ctrl/Cmd+Enter or the play button runs. c.source never contains the '#| export' pragma -- see Cell.export / the bookmark toggle in cell_toolbar."
    ta = Textarea(c.source, name='source', id=f'ta-{c.id}',
                  rows=str(max(2, c.source.count(chr(10)) + 1)),
                  cls='textarea textarea-bordered w-full font-mono',
                  data_cm='code', data_cid=str(c.id))
    parts = [Form(ta, hx_post=save_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML')]
    if c.output:
        parts += render_output_blocks(c.output)
    return Div(*parts)


In [ ]:
#| export
def render_cell(c:Cell, oob=None) -> FT:
    "Every cell type renders statically and opens its editor on click; only the actively-edited cell gets a live widget (CodeMirror for code, a plain textarea otherwise). `oob` is forwarded to `_cell_outer` -- see there."
    if c.ctype == 'code':
        return _cell_outer(c, cell_header(c), code_editor(c) if c.id == nb.selected else code_view(c), oob=oob)
    body = cell_body(c)
    body = Div(body, cls='cursor-text',
                   hx_get=edit_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML')
    return _cell_outer(c, cell_header(c), body, oob=oob)


In [ ]:
#| export
def _oob(spec:str, *content) -> FT:
    "Wrap `content` in a throwaway div carrying a positional htmx out-of-band directive (e.g. 'afterend:#cell-5' or 'beforeend:#notebook'). Needed because htmx's positional OOB swaps (beforebegin/afterend/beforeend) insert only the *children* of the OOB-tagged element, not the element itself -- so anything that needs to land in the DOM with its own id intact (a cell div, a pending placeholder) must be nested one level below the OOB wrapper, not carry the hx-swap-oob attribute itself."
    return Div(*content, hx_swap_oob=spec)

In [ ]:
#| export
def render_cell_edit(c:Cell) -> FT:
    "Inline editor. Code cells use CodeMirror (Python highlight, no wrap); notes/raw use a textarea. Shift/Ctrl/Cmd+Enter saves."
    if c.ctype == 'code':
        return _cell_outer(c, cell_header(c), code_editor(c))
    ta = Textarea(c.source, name='source', id=f'ta-{c.id}',
                  rows=str(max(3, c.source.count(chr(10)) + 2)),
                  cls='textarea textarea-bordered w-full font-mono',
                  data_cm='edit', data_cid=str(c.id))
    buttons = Div(Button('Save', type='button', cls='btn btn-primary btn-xs',
                         onclick=f'boopSave({c.id})'),
                  Button('Cancel', type='button', cls='btn btn-ghost btn-xs',
                         hx_get=view_cell.to(id=c.id), hx_target=f'#cell-{c.id}', hx_swap='outerHTML'),
                  cls='flex gap-2 justify-end mt-1')
    form = Form(ta, buttons, hx_post=save_cell.to(id=c.id),
                hx_target=f'#cell-{c.id}', hx_swap='outerHTML')
    return _cell_outer(c, cell_header(c), form)

In [ ]:
#| export
def render_nb() -> FT:
    "Render every cell in the notebook, in order, inside the #notebook container div."
    return Div(*[render_cell(c) for c in nb.cells], id='notebook', cls='flex flex-col')

## Composer

The bottom bar: type tabs, a textarea, Submit. Picking a tab sets the type
server-side; Submit creates the cell (running it, if code; spawning an Assistant
reply, if prompt).

In [ ]:
#| export
def composer(draft:str='', oob:bool=False) -> FT:
    "The bottom-of-page input bar: type tabs, a source textarea, and a Boop (submit) button."
    tabs = [fh.A(t.capitalize(),
                 cls=f'tab {"tab-active" if nb.compose_type==t else ""}',
                 hx_post=set_type.to(t=t), hx_target='#composer', hx_swap='outerHTML')
            for t in CTYPES]
    ta_kw = {'data_cm':'composer'} if nb.compose_type=='code' else {}
    div_kw = {'hx_swap_oob':'true'} if oob else {}
    return Div(
        Div(*tabs, cls='tabs tabs-boxed'),
        Form(Textarea(draft, placeholder=f'{nb.compose_type} cell…', name='source',
                      id='compose-input', rows='3',
                      onkeydown="if((event.shiftKey||event.ctrlKey||event.metaKey)&&event.key==='Enter')"
                               "{event.preventDefault();this.form.requestSubmit();}",
                      cls='textarea textarea-bordered w-full font-mono', **ta_kw),
             Div(Button('Boop', type='button', onclick='boopComposerSubmit()', cls='btn btn-primary'), cls='flex justify-end mt-2'),
             hx_post=submit_cell, hx_target='#notebook', hx_swap='beforeend'),
        id='composer', cls='border-t border-base-300 pt-3 mt-4', **div_kw)


In [ ]:
#| export
def render_app(draft:str='') -> FT:
    "The whole notebook view: all cells plus the composer, wrapped in one container div."
    return Div(render_nb(), composer(draft), id='app')

## Routes

Composer/toolbar routes plus the command-mode routes driven by hotkeys
(`select`, `select_delta`, `insert`, `del_selected`, `settype_selected`).

In [ ]:
#| export
# ---- top menu / control bar ----
def theme_swap() -> FT:
    "DaisyUI sun/moon swap; drives boopApplyTheme (default dark)."
    # tooltip-right-align: this is the rightmost element in the navbar, flush against the viewport
    # edge, so a centered tooltip-bottom would overflow off-screen -- anchor its right edge instead.
    return NotStr('<label class="swap swap-rotate btn btn-ghost btn-circle btn-sm tooltip tooltip-bottom tooltip-right-align" data-tip="Toggle light/dark">'
      '<input type="checkbox" id="theme-toggle" onchange="boopThemeToggle(this)" checked />'
      '<svg class="swap-off h-5 w-5 fill-current" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
      '<path d="M5.64,17l-.71.71a1,1,0,0,0,0,1.41,1,1,0,0,0,1.41,0l.71-.71A1,1,0,0,0,5.64,17ZM5,12a1,1,0,0,0-1-1H3a1,1,0,0,0,0,2H4A1,1,0,0,0,5,12Zm7-7a1,1,0,0,0,1-1V3a1,1,0,0,0-2,0V4A1,1,0,0,0,12,5ZM5.64,7.05a1,1,0,0,0,.7.29,1,1,0,0,0,.71-.29,1,1,0,0,0,0-1.41l-.71-.71A1,1,0,0,0,4.93,6.34Zm12,.29a1,1,0,0,0,.7-.29l.71-.71a1,1,0,1,0-1.41-1.41L17,5.64a1,1,0,0,0,0,1.41A1,1,0,0,0,17.66,7.34ZM21,11H20a1,1,0,0,0,0,2h1a1,1,0,0,0,0-2Zm-9,8a1,1,0,0,0-1,1v1a1,1,0,0,0,2,0V20A1,1,0,0,0,12,19ZM18.36,17A1,1,0,0,0,17,18.36l.71.71a1,1,0,0,0,1.41,0,1,1,0,0,0,0-1.41ZM12,6.5A5.5,5.5,0,1,0,17.5,12,5.51,5.51,0,0,0,12,6.5Zm0,9A3.5,3.5,0,1,1,15.5,12,3.5,3.5,0,0,1,12,15.5Z"/></svg>'
      '<svg class="swap-on h-5 w-5 fill-current" xmlns="http://www.w3.org/2000/svg" viewBox="0 0 24 24">'
      '<path d="M21.64,13a1,1,0,0,0-1.05-.14,8.05,8.05,0,0,1-3.37.73A8.15,8.15,0,0,1,9.08,5.49a8.59,8.59,0,0,1,.25-2A1,1,0,0,0,8,2.36,10.14,10.14,0,1,0,22,14.05,1,1,0,0,0,21.64,13Zm-9.5,6.69A8.14,8.14,0,0,1,7.08,5.22v.27A10.15,10.15,0,0,0,17.22,15.63a9.79,9.79,0,0,0,2.1-.22A8.11,8.11,0,0,1,12.14,19.73Z"/></svg></label>')


NameError: name 'rt' is not defined

In [ ]:
#| export
_BOOP2NB = {'code':'code', 'note':'markdown', 'prompt':'markdown', 'raw':'raw', 'assistant':'markdown'}  # our ctype -> nbformat cell_type

In [ ]:
#| export
def _blocks_to_nb_outputs(blocks:list[dict]) -> list:
    "Cell.output's block list -> real nbformat outputs, so saved .ipynb files stay valid (and render in GitHub/real Jupyter too)."
    outs = []
    for b in blocks:
        if b['type'] == 'stream':
            outs.append(_nbf.v4.new_output('stream', name='stdout', text=b['data']))
        elif b['type'] == 'error':
            ename, _, evalue = b['data'].partition(': ')
            outs.append(_nbf.v4.new_output('error', ename=ename, evalue=evalue, traceback=[b['data']]))
        else:  # display
            outs.append(_nbf.v4.new_output('display_data', data={b['mime']: b['data']}))
    return outs

def _nb_outputs_to_blocks(outputs:list) -> list[dict]:
    "Inverse of _blocks_to_nb_outputs()."
    blocks = []
    for o in outputs:
        ot = o.get('output_type')
        if ot == 'stream':
            blocks.append({'type':'stream', 'mime':None, 'data':o.get('text', '')})
        elif ot == 'error':
            data = f"{o.get('ename','')}: {o.get('evalue','')}" if o.get('ename') else o.get('evalue', '')
            blocks.append({'type':'error', 'mime':None, 'data':data})
        elif ot in ('display_data', 'execute_result'):
            data = o.get('data', {})
            mime = next((m for m in _MIME_PRIORITY if m in data), next(iter(data), None))
            if mime: blocks.append({'type':'display', 'mime':mime, 'data':data[mime]})
    return blocks

In [ ]:
#| export
def save_notebook(path:str|Path|None=None) -> Path:
    "Serialize `nb.cells` to a real Jupyter notebook file (`{nb.name}.ipynb` in the cwd, by default). Preserves each cell's original .ipynb id (or captures a freshly-assigned one) so unchanged cells don't produce noisy git diffs. The '#| export' pragma is prepended to code cell source here (and only here) -- nbdev's own parser needs it literally in the file, but boopiter keeps it out of c.source/the editor; see load_notebook() for the inverse."
    path = Path(path) if path else Path.cwd()/f'{nb.name}.ipynb'
    doc = _nbf.v4.new_notebook()
    for c in nb.cells:
        meta = {'boopiter': {'ctype': c.ctype, 'visible': c.visible}}
        kind = _BOOP2NB.get(c.ctype, 'raw')
        idkw = {'id': c.nb_id} if c.nb_id else {}
        src = f'#| export\n{c.source}' if (c.ctype == 'code' and c.export) else c.source
        if kind == 'code':
            outputs = _blocks_to_nb_outputs(c.output) if c.output else []
            cell = _nbf.v4.new_code_cell(src, outputs=outputs, metadata=meta, **idkw)
        elif kind == 'markdown':
            cell = _nbf.v4.new_markdown_cell(src, metadata=meta, **idkw)
        else:
            cell = _nbf.v4.new_raw_cell(src, metadata=meta, **idkw)
        c.nb_id = cell['id']  # capture the (possibly just-generated) id so future saves reuse it too
        doc.cells.append(cell)
    _nbf.write(doc, str(path))
    return path


In [ ]:
#| export
_NB_FALLBACK = {'code':'code', 'markdown':'note', 'raw':'raw'}  # nbformat cell_type -> our ctype, for plain (non-boopiter) notebooks

In [ ]:
#| export
def load_notebook(path:str|Path) -> Notebook:
    "Load a Jupyter notebook file into `nb`, replacing its current contents. Inverse of save_notebook() -- detects a leading '#| export' line on code cells, sets c.export, and strips it out of the stored/displayed source."
    path = Path(path)
    doc = _nbf.read(str(path), as_version=4)
    nb.cells.clear()
    nb._nid = 0
    nb.selected = None
    for cell in doc.cells:
        meta = cell.get('metadata', {}).get('boopiter', {})
        ctype = meta.get('ctype')
        if ctype not in CTYPES + ('assistant',):
            ctype = _NB_FALLBACK.get(cell.cell_type, 'raw')  # plain (non-boopiter) notebook
        output = None
        if ctype == 'code':
            output = _nb_outputs_to_blocks(cell.get('outputs', [])) or None
        src, exported = cell.source, False
        if ctype == 'code' and _has_export(src):
            exported, src = True, _strip_export(src)
        nb.add(ctype, src, output=output, visible=meta.get('visible', True), nb_id=cell.get('id'), export=exported)
    nb.name = str(path.with_suffix(''))  # keep the directory, only strip .ipynb
    return nb


In [ ]:
#| export
def fname_display() -> FT:
    "The clickable filename shown in the top bar; click to rename."
    return Span(nb.name, id='fname', data_tip='Click to rename',
                cls='cursor-pointer font-mono opacity-80 hover:opacity-100 tooltip tooltip-bottom',
                hx_get=rename_form, hx_target='#fname', hx_swap='outerHTML')


In [ ]:
#| export
@rt
def rename_form() -> FT:
    "Swap the filename display for a text input, focused and pre-selected, to rename the notebook."
    return Form(Input(value=nb.name, name='name',
                      cls='input input-sm input-bordered font-mono',
                      onkeydown="if(event.key===\'Escape\'){this.form.requestSubmit();}"),
                Script("var i=document.querySelector(\'#fname input\'); if(i){i.focus();i.select();}"),
                id='fname', hx_post=rename, hx_target='#fname', hx_swap='outerHTML')

In [ ]:
#| export
def model_dropdown() -> FT:
    "Select which LLM answers Prompt cells; selection lives on `nb.model`."
    if not nb.models:
        return Span('no models', cls='text-xs opacity-50 tooltip tooltip-bottom', data_tip='No local LLMs found (is Ollama running?)')
    opts = [fh.Option(m, value=m, selected=(m == nb.model)) for m in nb.models]
    return fh.Select(*opts, name='model', cls='select select-sm select-bordered',
                      hx_post=set_model, hx_trigger='change', hx_swap='none')


In [ ]:
#| export
@rt
def set_model(model:str) -> str:
    "Switch the active Prompt-answering model, stopping the previous Ollama model to free its VRAM."
    if model in nb.models and model != nb.model:
        old, nb.model = nb.model, model
        if old and old.startswith('ollama/'):
            import subprocess
            try: subprocess.run(['ollama', 'stop', old.removeprefix('ollama/')], capture_output=True, timeout=5)
            except Exception: pass
    return ''

In [ ]:
#| export
def file_menu() -> FT:
    "Hamburger dropdown: New / Open (file browser) / Save / Download / Restart Server."
    items = [
        Li(fh.A('New', href=new_notebook.to(),
                onclick="return confirm('Discard the current notebook and start a new one?')")),
        Li(fh.A('Open', onclick="document.getElementById('file-modal').showModal()",
                hx_get=browse.to(), hx_target='#file-browser-body', hx_swap='innerHTML')),
        Li(fh.A('Save', href='javascript:void(0)', onclick='boopSaveNotebook()')),
        Li(fh.A('Download', href=download.to())),
        Li(fh.A('Restart Server', href='javascript:void(0)', onclick='boopRestartServer()')),
    ]
    return Div(
        Div(Icon('bars-3'), tabindex='0', role='button', cls='btn btn-ghost btn-circle btn-sm'),
        Ul(*items, tabindex='0', cls='dropdown-content menu bg-base-200 rounded-box z-10 w-40 p-1 shadow'),
        cls='dropdown dropdown-bottom')

In [ ]:
#| export
def file_browser_modal() -> FT:
    "The (initially empty/hidden) dialog that hosts the file-browser listing, opened by file_menu()'s Open item."
    return Dialog(
        Div(
            Div('Open notebook', cls='font-semibold mb-2'),
            Div(id='file-browser-body'),
            Div(Form(fh.Button('Close', cls='btn btn-sm'), method='dialog'), cls='modal-action'),
            cls='modal-box'),
        # DaisyUI's click-outside-to-close pattern: a full-screen backdrop form whose submit
        # (triggered by any click on it, since it has no other interactive content) just closes
        # the <dialog> via method='dialog', same as the Close button.
        Form(fh.Button('close', cls='cursor-default'), method='dialog', cls='modal-backdrop'),
        id='file-modal', cls='modal')


In [ ]:
#| export
def help_modal() -> FT:
    "Keyboard-shortcuts cheat sheet, opened by the '?' button in the top bar. Same modal/backdrop pattern as file_browser_modal()."
    command_mode = [
        ('j / ↓ , k / ↑', 'Select the next / previous cell'),
        ('a', 'Insert a new cell above the selection'),
        ('b', 'Insert a new cell below the selection'),
        ('y', 'Change the selected cell to Code'),
        ('m', 'Change the selected cell to Note'),
        ('r', 'Change the selected cell to Raw'),
        ('c', 'Copy the selected cell'),
        ('x', 'Cut the selected cell'),
        ('v', 'Paste after the selected cell'),
        ('d d', 'Delete the selected cell (press d twice)'),
        ('s', 'Save the notebook'),
    ]
    editing_mode = [
        ('Shift / Ctrl / Cmd + Enter', 'Save (and run) the cell'),
        ('Ctrl / Cmd + /', 'Toggle line comments'),
        ('Shift + Ctrl / Cmd + -', 'Split the cell at the cursor'),
    ]
    def section(title, pairs):
        # kbd-sm rendered visibly smaller than the description text next to it -- drop the size
        # modifier (DaisyUI's default kbd size) and match font-size explicitly to the description.
        return [Div(title, cls='text-xs font-semibold opacity-60 mt-3 mb-1 first:mt-0')] + [
            Div(Kbd(k, cls='kbd text-sm'), Span(v, cls='text-sm'), cls='flex items-center gap-3 py-1') for k, v in pairs]
    return Dialog(
        Div(
            Div('Keyboard shortcuts', cls='font-semibold mb-2'),
            *section('Command mode (click a cell to select it)', command_mode),
            *section('While editing a cell', editing_mode),
            Div(Form(fh.Button('Close', cls='btn btn-sm'), method='dialog'), cls='modal-action'),
            cls='modal-box'),
        Form(fh.Button('close', cls='cursor-default'), method='dialog', cls='modal-backdrop'),
        id='help-modal', cls='modal')


In [ ]:
#| export
def top_bar() -> FT:
    "The whole navbar: file menu + logo + filename on the left, model picker + kernel/theme controls on the right."
    brand = Div(file_menu(), Img(src='/logo.png', cls='h-8 w-8 rounded-full'),
                Span('boopiter', cls='font-bold text-lg'),
                Span('/', cls='opacity-40'), fname_display(),
                cls='flex items-center gap-2')
    # 19px landed between size-4 (16px, too small next to the moon/sun) and size-5 (20px, ended up
    # looking bigger than the moon/sun -- these are stroked outline icons vs. theme_swap()'s filled
    # ones, so the same box reads heavier); 18px still read as closer to the smaller end.
    icon_cls = 'size-[19px]'
    ctrls = Div(
        model_dropdown(),
        fh.Button(Icon('question-mark-circle', cls=icon_cls), data_tip='Keyboard shortcuts', cls='btn btn-ghost btn-circle btn-sm tooltip tooltip-bottom',
                  onclick="document.getElementById('help-modal').showModal()"),
        fh.Button(Icon('x-circle', cls=icon_cls), data_tip='Interrupt kernel', cls='btn btn-ghost btn-circle btn-sm tooltip tooltip-bottom',
                  hx_post=interrupt_kernel, hx_swap='none'),
        fh.Button(Icon('arrow-path', cls=icon_cls), data_tip='Restart kernel', cls='btn btn-ghost btn-circle btn-sm tooltip tooltip-bottom',
                  hx_post=restart_kernel, hx_swap='none'),
        fh.Button(Icon('play-circle', cls=icon_cls), data_tip='Run all code cells', cls='btn btn-ghost btn-circle btn-sm tooltip tooltip-bottom',
                  hx_post=run_all, hx_target='#notebook', hx_swap='outerHTML'),
        theme_swap(), cls='flex items-center gap-1')
    return Div(brand, ctrls, file_browser_modal(), help_modal(),
               cls='navbar bg-base-200 shadow px-4 flex justify-between shrink-0')


In [ ]:
#| export
@rt('/_boopiter_ping')
def boopiter_ping() -> str:
    "Identity check so `boopiter launch` can tell a live boopiter instance apart from something else on the port."
    return 'boopiter'

In [ ]:
#| export
@rt('/logo.png')
def logo_png() -> FileResponse:
    "Serve the boopiter logo, used both as favicon and in the top bar."
    return FileResponse(Path(__file__).parent.parent/'images/logo.png')

In [ ]:
#| export
@rt('/tailwind.css')
def tailwind_css() -> FileResponse:
    "Serve the precompiled Tailwind CSS built by _build_tailwind() at launch."
    return FileResponse(Path(__file__).parent/'static/tailwind.css')

In [ ]:
#| export
@rt
def index() -> tuple:
    "The full page: title, top bar, the notebook+composer, and the save-toast slot."
    return (Title('boopiter'),
            Div(top_bar(),
                Div(Div(render_app(), cls='max-w-3xl mx-auto p-4'),
                    cls='flex-1 overflow-y-auto'),
                Div(id='save-toast', cls='toast toast-top toast-end z-50'),
                cls='h-screen flex flex-col'))

In [ ]:
#| export
def _toast(msg:str, ok:bool=True) -> FT:
    "A little 'Saved' (or error) notice, out-of-band-swapped into #save-toast, that clears itself after ~1.8s."
    return Div(
        Div(msg, cls=f"alert {'alert-success' if ok else 'alert-error'} shadow-lg text-sm py-2 px-4"),
        Script("setTimeout(function(){ var t=document.getElementById('save-toast'); if(t) t.innerHTML=''; }, 1800)"),
        id='save-toast', cls='toast toast-top toast-end z-50', hx_swap_oob='true')

In [ ]:
#| export
@rt
def save_now() -> FT:
    "Save the notebook to disk and show a toast confirming success (or failure)."
    try:
        p = save_notebook()
        return _toast(f'Saved {p.name}')
    except Exception as e:
        return _toast(f'Save failed: {e}', ok=False)

In [ ]:
#| export
def _safe_dir(path:str|None) -> Path:
    "Resolve `path` (relative to BROWSE_ROOT) and clamp it back to BROWSE_ROOT if it tries to escape (e.g. via '..')."
    cur = (BROWSE_ROOT/(path or '')).resolve()
    if cur != BROWSE_ROOT and BROWSE_ROOT not in cur.parents: cur = BROWSE_ROOT
    if not cur.is_dir(): cur = BROWSE_ROOT
    return cur

In [ ]:
#| export
@rt
def browse(path:str|None=None) -> FT:
    "Jupyter-tree-style directory listing for the file-browser modal, rooted at BROWSE_ROOT."
    cur = _safe_dir(path)
    rel = cur.relative_to(BROWSE_ROOT)
    entries = sorted(cur.iterdir(), key=lambda p: (p.is_file(), p.name.lower()))
    # Reuse DaisyUI's own `menu` component (same as file_menu()'s hamburger dropdown, which also
    # skips the `-sm` size modifier) so the highlight color and font size both match exactly.
    row_cls = 'flex items-center gap-2 py-1 px-2'
    rows = []
    if cur != BROWSE_ROOT:
        up = '' if rel.parent == Path('.') else str(rel.parent)
        rows.append(Li(fh.A(Icon('folder'), ' ..', hx_get=browse.to(path=up),
                             hx_target='#file-browser-body', cls=row_cls)))
    for p in entries:
        if p.name.startswith('.'): continue
        relp = str(p.relative_to(BROWSE_ROOT))
        if p.is_dir():
            rows.append(Li(fh.A(Icon('folder'), ' ' + p.name, hx_get=browse.to(path=relp),
                                 hx_target='#file-browser-body', cls=row_cls)))
        elif p.suffix == '.ipynb':
            rows.append(Li(fh.A(Icon('document-text'), ' ' + p.name, href=open_file.to(path=relp), cls=row_cls)))
        else:
            rows.append(Li(Div(Icon('document-text'), Span(p.name), cls=row_cls + ' opacity-40'),
                            cls='pointer-events-none'))
    return Div(Div('/' if str(rel) == '.' else f'/{rel}', cls='font-mono text-xs opacity-60 mb-2'),
               Ul(*rows, cls='menu p-0'), id='file-browser-body')


In [ ]:
#| export
@rt
def open_file(path:str) -> RedirectResponse:
    "Open a notebook found by the file browser. Redirects to / afterward so the address bar (and thus a later page reload) doesn't stay pinned to this action -- reloading /open_file?path=... would silently re-run the load and discard any unsaved edits made since."
    target = _safe_dir(Path(path).parent)/Path(path).name
    try: load_notebook(target.relative_to(BROWSE_ROOT))  # relative, to match the CLI's nb.name style
    except Exception: pass
    return RedirectResponse('/', status_code=303)

In [ ]:
#| export
@rt
def new_notebook() -> RedirectResponse:
    "Discard the current notebook and start a blank one. Redirects to / afterward -- same reasoning as open_file()."
    nb.reset()
    return RedirectResponse('/', status_code=303)

In [ ]:
#| export
@rt
def restart_server() -> str:
    "nbdev-export the notebooks as a real (blocking) subprocess -- so we get a genuine exit code instead of guessing a delay -- then clear __pycache__ (WSL/Windows-mounted filesystems can have coarse mtime resolution, which can trick Python into serving stale cached bytecode right after a fast save-then-restart) and exec a fresh copy of this process (same PID, same port). Aborts (leaves the current process running) if export fails. Does NOT save the notebook first; Save manually beforehand if you want to keep unsaved edits."
    def _do_restart() -> None:
        "Runs in a background thread: exports, clears the bytecode cache, and execs a fresh process."
        time.sleep(0.3)  # let the HTTP response reach the browser before we touch anything disruptive
        exe = Path(sys.executable).parent/'nbdev-export'
        exe = str(exe) if exe.exists() else 'nbdev-export'
        result = subprocess.run([exe], capture_output=True, text=True)
        if result.returncode != 0:
            print(f'nbdev-export failed ({result.returncode}), aborting restart:\n{result.stderr}', file=sys.stderr, flush=True)
            return
        cache_dir = Path(__file__).parent/'__pycache__'
        if cache_dir.exists(): shutil.rmtree(cache_dir, ignore_errors=True)
        port = os.environ.get('BOOPITER_PORT', '8000')
        args = [sys.executable, sys.argv[0]]
        if nb.name != 'untitled': args.append(f'{nb.name}.ipynb')
        args += ['--port', port]
        os.execv(sys.executable, args)
    threading.Thread(target=_do_restart, daemon=True).start()
    return ''

In [ ]:
#| export
@rt
def download() -> FileResponse:
    "Save the notebook, then send it to the browser as a file download."
    p = save_notebook()
    return FileResponse(str(p), filename=p.name)

In [ ]:
#| export
@rt
def rename(name:str) -> FT:
    "Rename and persist the notebook to `{new_name}.ipynb` in the server's cwd."
    nb.name = name.strip() or nb.name
    save_notebook()
    return fname_display()

In [ ]:
#| export
@rt
def restart_kernel() -> str:
    "Reset the shared IPython shell's namespace, clearing all user-defined variables/functions."
    _shell.reset()          # clear kernel namespace
    return ''

In [ ]:
#| export
@rt
def run_all() -> FT:
    "Run every Code cell, top to bottom, in place (Prompt/Note/Raw/Assistant cells are left untouched)."
    for c in nb.cells:
        if c.ctype == 'code': c.output = run_code(c.source)
    return render_nb()

In [ ]:
#| export
@rt
def interrupt_kernel() -> str:
    "Best-effort interrupt: inject a KeyboardInterrupt into the currently-running background thread, if any (CPython's PyThreadState_SetAsyncExc, the same low-level trick real kernels use for SIGINT-based interrupts). It fires at the interrupted thread's next bytecode boundary, so it reliably breaks ordinary Python loops (e.g. a tqdm-wrapped for-loop) but can't preempt a single blocking C call already in flight."
    st = _run_state
    if st is not None and st.thread is not None and st.thread.is_alive():
        import ctypes
        tid = ctypes.c_long(st.thread.ident)
        res = ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, ctypes.py_object(KeyboardInterrupt))
        if res > 1:  # affected more than one thread -- back it out
            ctypes.pythonapi.PyThreadState_SetAsyncExc(tid, None)
    return ''


In [ ]:
#| export
@rt
def set_type(t:str) -> FT:
    "Change the composer's current cell type (what a new cell becomes when you hit Boop)."
    if t in CTYPES: nb.compose_type = t
    return composer()

In [ ]:
#| export
def run_prompt_cell(id:int) -> Cell|None:
    "(Re)send `id`'s prompt -- plus everything visible above it -- to the LLM; create or refresh the paired Assistant cell. Returns that cell."
    c = nb.get(id)
    if not c or c.ctype != 'prompt': return None
    if nb.model:
        try: reply = prompt_llm(llm_context(nb, c.id), model=nb.model, tools=nb.tools)
        except Exception as e: reply = f'(error calling {nb.model}: {e})'
    else:
        reply = stub_reply(nb, c.source)
    i = nb.index(c.id)
    nxt = nb.cells[i+1] if i+1 < len(nb.cells) else None
    if nxt is not None and nxt.ctype == 'assistant':
        nxt.source, nxt.model = reply, nb.model
        nxt.ts = datetime.now().strftime('%I:%M:%S %p')
    else:
        nxt = nb.insert_at(i+1, 'assistant', reply, model=nb.model)
    return nxt

In [ ]:
#| export
def pending_cell(prompt_id:int, oob_swap:str=None) -> FT:
    "Placeholder shown right after a prompt is submitted; hx-trigger=load immediately fires the real (slow) LLM call and swaps itself out for the real Assistant cell once it replies. If `oob_swap` is given (an htmx hx-swap-oob spec like 'outerHTML:#cell-5' or 'afterend:#cell-5'), this placeholder is delivered out-of-band at that location instead of being the response's main swap target."
    kw = {'hx_swap_oob': oob_swap} if oob_swap else {}
    return Div('Assistant: Tricky...', id=f'pending-{prompt_id}',
               cls='border-l-4 border-error pl-3 py-2 my-2 ml-8 opacity-60 italic',
               hx_post=run_prompt_pending.to(id=prompt_id), hx_target=f'#pending-{prompt_id}',
               hx_swap='outerHTML', hx_trigger='load', **kw)


In [ ]:
#| export
@rt
def run_prompt_pending(id:int) -> FT|str:
    "Actually run a Prompt's pending LLM call and return the real Assistant cell, replacing the pending_cell() placeholder."
    c2 = run_prompt_cell(id)
    return render_cell(c2) if c2 else ''

In [ ]:
#| export
def add_cell(t:str, source:str) -> list[Cell]:
    "Create a cell of type `t`: run it if code, just create it if prompt (its Assistant reply is added asynchronously via pending_cell). Returns the new cell(s)."
    if t == 'code':
        return [nb.add('code', source, output=run_code(source))]
    elif t == 'prompt':
        return [nb.add('prompt', source)]
    else:
        return [nb.add(t, source)]

In [ ]:
#| export
@rt
def submit_cell(source:str) -> tuple:
    "Append only the new cell(s) to #notebook and reset the composer out-of-band, so untouched cells' editors are never re-created. A just-submitted Prompt gets a 'Thinking...' placeholder that fetches its own reply."
    new = add_cell(nb.compose_type, source) if source.strip() else []
    pending = [pending_cell(new[-1].id)] if new and nb.compose_type == 'prompt' else []
    return *[render_cell(c) for c in new], *pending, composer(oob=True)

In [ ]:
#| export
@rt
def split(source:str, pos:int) -> tuple:
    "Split the composer at the caret: head becomes a cell, tail stays in the composer."
    head, tail = source[:pos], source[pos:]
    new = add_cell(nb.compose_type, head) if head.strip() else []
    pending = [pending_cell(new[-1].id)] if new and nb.compose_type == 'prompt' else []
    return *[render_cell(c) for c in new], *pending, composer(draft=tail, oob=True)

In [ ]:
#| export
@rt
def split_cell(id:int, pos:int) -> FT|tuple:
    "Split an existing cell at caret position `pos`: it keeps the text before the cursor; a new cell of the same type (and, for code, the same export flag) is inserted right after it with the text after the cursor. Neither half is (re-)executed. Blank lines right at the split point (e.g. the PEP8 spacer between two functions) are trimmed off the boundary -- otherwise the new cell would start with an ugly-looking leading blank line."
    c = nb.get(id)
    if not c: return ''
    head, tail = c.source[:pos], c.source[pos:]
    c.source = head.rstrip('\n')
    i = nb.index(c.id)
    new = nb.insert_at(i+1, c.ctype, tail.lstrip('\n'), export=c.export)
    nb.selected = new.id
    return render_cell(c), _oob(f'afterend:#cell-{c.id}', render_cell(new))


In [ ]:
#| export
@rt
def run_cell(id:int) -> FT:
    "Run this cell (Code: kick off background execution and return a placeholder that streams its progress; Prompt: return a pending placeholder that fetches its own reply), so the caller can target just this cell instead of the whole notebook."
    c = nb.get(id)
    if c and c.ctype == 'code':
        return _start_code_run(c)
    elif c and c.ctype == 'prompt':
        return pending_cell(id)
    return render_nb()


In [ ]:
#| export
@rt
def toggle_vis(id:int) -> FT|str:
    "Toggle whether this cell is visible to the LLM (shown/hidden in llm_context)."
    c = nb.get(id)
    if not c: return ''
    c.visible = not c.visible
    return render_cell(c)


In [ ]:
#| export
@rt
def toggle_export(id:int) -> FT:
    "Toggle a code cell's '#| export' flag (the bookmark icon in cell_toolbar)."
    c = nb.get(id)
    if c and c.ctype == 'code':
        c.export = not c.export
    return render_cell(c) if c else render_nb()

In [ ]:
#| export
@rt
def del_cell(id:int) -> tuple:
    "Delete this cell (or its Prompt+Assistant pair). Removes just the affected DOM node(s) out-of-band instead of re-rendering the whole notebook."
    rng = nb.pair_range(id)
    if rng is None: return ('',)
    ids = [nb.cells[j].id for j in range(rng[0], rng[1]+1)]
    nb.remove(id)
    return tuple(Div(id=f'cell-{rid}', hx_swap_oob='delete') for rid in ids)


In [ ]:
#| export
@rt
def move_cell(id:int, delta:int) -> tuple:
    "Move this cell (or its Prompt+Assistant pair) up (delta=-1) or down (delta=1). Rather than re-rendering the whole notebook, this deletes the moved block's DOM node(s) out-of-band and reinserts them next to whichever cell/pair they swapped with -- the rest of the notebook's DOM (and the page's scroll position) is never touched."
    rng = nb.pair_range(id)
    if rng is None: return ('',)
    lo, hi = rng
    if delta < 0:
        if lo == 0: return ('',)
        nlo, _ = nb.pair_range(nb.cells[lo-1].id)
        anchor_id, swap = nb.cells[nlo].id, 'beforebegin'
    elif delta > 0:
        if hi >= len(nb.cells) - 1: return ('',)
        _, nhi = nb.pair_range(nb.cells[hi+1].id)
        anchor_id, swap = nb.cells[nhi].id, 'afterend'
    else:
        return ('',)
    block_ids = [nb.cells[j].id for j in range(lo, hi+1)]
    nb.move(id, delta)
    deletes = [Div(id=f'cell-{bid}', hx_swap_oob='delete') for bid in block_ids]
    order = reversed(block_ids) if swap == 'afterend' else block_ids  # afterend inserts stack in reverse of iteration order
    inserts = [_oob(f'{swap}:#cell-{anchor_id}', render_cell(nb.get(bid))) for bid in order]
    return (*deletes, *inserts)


In [ ]:
#| export
# --- command-mode (hotkey) routes ---
@rt
def select(id:int) -> FT|tuple|str:
    "Select this cell (highlights it and anchors j/k navigation). Updates just the newly- and previously-selected cells, not the whole notebook."
    old, c = nb.selected, nb.get(id)
    if c is None: return ''
    nb.selected = id
    old_c = nb.get(old) if (old is not None and old != id) else None
    return (render_cell(c), render_cell(old_c, oob=True)) if old_c is not None else render_cell(c)


In [ ]:
#| export
@rt
def select_delta(delta:int) -> tuple:
    "Move the selection up (delta=-1) or down (delta=1) -- the j/k hotkeys. Triggered by a global hotkey (not a specific cell's button), so the response carries out-of-band updates for whichever cells actually changed rather than a full re-render."
    old = nb.selected
    if nb.cells:
        i = nb.sel_index()
        i = (0 if delta > 0 else len(nb.cells)-1) if i is None else min(max(i+delta, 0), len(nb.cells)-1)
        nb.selected = nb.cells[i].id
    changed = {old, nb.selected} - {None}
    parts = [render_cell(c, oob=True) for cid in changed if (c := nb.get(cid)) is not None]
    return tuple(parts) if parts else ('',)


In [ ]:
#| export
@rt
def insert(where:str) -> tuple:
    "Insert a new blank cell above or below the current selection -- the a/b hotkeys. Inserts the new cell out-of-band next to its anchor, and refreshes the previously-selected cell (to drop its highlight ring) rather than re-rendering the whole notebook."
    old_sel = nb.selected
    i = nb.sel_index()
    if i is None:
        new = nb.insert_at(len(nb.cells), nb.compose_type, '')
        nb.selected = new.id
        return (_oob('beforeend:#notebook', render_cell(new)),)
    anchor_id = nb.cells[i].id
    pos = i if where == 'above' else i+1
    new = nb.insert_at(pos, nb.compose_type, '')
    nb.selected = new.id
    swap = 'beforebegin' if where == 'above' else 'afterend'
    parts = [_oob(f'{swap}:#cell-{anchor_id}', render_cell(new))]
    old_c = nb.get(old_sel) if old_sel != nb.selected else None
    if old_c is not None: parts.append(render_cell(old_c, oob=True))
    return tuple(parts)


In [ ]:
#| export
@rt
def del_selected() -> tuple:
    "Delete the currently-selected cell (or its Prompt+Assistant pair) -- the d-d hotkey."
    if nb.selected is None: return ('',)
    i = nb.sel_index()
    rng = nb.pair_range(nb.selected)
    ids = [nb.cells[j].id for j in range(rng[0], rng[1]+1)] if rng else [nb.selected]
    nb.remove(nb.selected)
    nb.selected = nb.cells[min(i, len(nb.cells)-1)].id if nb.cells else None
    parts = [Div(id=f'cell-{rid}', hx_swap_oob='delete') for rid in ids]
    new_c = nb.get(nb.selected) if nb.selected is not None else None
    if new_c is not None: parts.append(render_cell(new_c, oob=True))
    return tuple(parts)


In [ ]:
#| export
@rt
def cut_selected() -> tuple:
    "Cut the currently-selected cell (or its pair) to the clipboard -- the x hotkey."
    if nb.selected is None: return ('',)
    rng = nb.pair_range(nb.selected)
    ids = [nb.cells[j].id for j in range(rng[0], rng[1]+1)] if rng else [nb.selected]
    nb.cut_range(nb.selected)  # also updates nb.selected
    parts = [Div(id=f'cell-{rid}', hx_swap_oob='delete') for rid in ids]
    new_c = nb.get(nb.selected) if nb.selected is not None else None
    if new_c is not None: parts.append(render_cell(new_c, oob=True))
    return tuple(parts)


In [ ]:
#| export
@rt
def copy_selected() -> str:
    "Copy the currently-selected cell (or its pair) to the clipboard -- the c hotkey."
    if nb.selected is not None: nb.copy_range(nb.selected)
    return ''

In [ ]:
#| export
@rt
def paste_selected() -> tuple:
    "Paste the clipboard after the currently-selected cell -- the v hotkey. Inserts the pasted cell(s) out-of-band, right after their anchor (or at the end, if nothing was selected), instead of re-rendering the whole notebook."
    old_sel = nb.selected
    i = nb.sel_index()
    anchor_id = nb.cells[i].id if i is not None else None
    pasted = nb.paste_after(nb.selected)  # also updates nb.selected
    if not pasted: return ('',)
    parts = []
    if anchor_id is None:
        for c in pasted: parts.append(_oob('beforeend:#notebook', render_cell(c)))
    else:
        prev = anchor_id
        for c in pasted:
            parts.append(_oob(f'afterend:#cell-{prev}', render_cell(c)))
            prev = c.id
    old_c = nb.get(old_sel) if old_sel != nb.selected else None
    if old_c is not None: parts.append(render_cell(old_c, oob=True))
    return tuple(parts)


In [ ]:
#| export
@rt
def settype_selected(t:str) -> FT|str:
    "Change the currently-selected cell's type -- the m/y/r hotkeys."
    c = nb.get(nb.selected) if nb.selected is not None else None
    if not (c and t in CTYPES): return ''
    c.ctype = t
    return render_cell(c, oob=True)


In [ ]:
#| export
@rt
def set_ctype(id:int, t:str) -> FT:
    "Change one cell's type in place; returns just that cell so the rest of the notebook is untouched."
    c = nb.get(id)
    if c and t in CTYPES and t != c.ctype:
        c.ctype = t
        c.output = None  # stale output no longer meaningful under the new type
    return render_cell(c) if c else render_nb()

In [ ]:
#| export
# --- inline editing ---
@rt
def edit_cell(id:int) -> FT:
    "Switch this cell into its live editor (CodeMirror for code, a plain textarea otherwise)."
    c = nb.get(id)
    if not c: return render_nb()
    nb.selected = id
    return render_cell_edit(c)

In [ ]:
#| export
@rt
def view_cell(id:int) -> FT:
    "Switch this cell back to its static (non-editing) view -- used by the Cancel button."
    c = nb.get(id)
    return render_cell(c) if c else render_nb()

In [ ]:
#| export
@rt
def save_cell(id:int, source:str) -> FT|tuple:
    "Commit an edited cell's source (running it if it's code, or re-prompting the LLM if it's a prompt). Targets just this cell (and, for prompts, the paired Assistant cell) rather than the whole notebook, so editing a cell deep in a long notebook doesn't blow away scroll position."
    c = nb.get(id)
    if not c: return render_nb()
    c.source = source
    if c.ctype == 'code':
        return _start_code_run(c)
    elif c.ctype == 'prompt':
        i = nb.index(c.id)
        nxt = nb.cells[i+1] if i+1 < len(nb.cells) else None
        if nxt is not None and nxt.ctype == 'assistant':
            # 'true'/'outerHTML' OOB swaps keep the tagged element itself, so pending_cell can carry the directive directly.
            pend = pending_cell(id, oob_swap=f'outerHTML:#cell-{nxt.id}')
        else:
            # Positional OOB swaps (afterend/beforebegin/beforeend) insert only the tagged element's
            # *children* -- so pending_cell must be wrapped, not itself carry the hx-swap-oob attribute,
            # or its own id/hx-trigger="load" would be discarded on insertion. See _oob().
            pend = _oob(f'afterend:#cell-{c.id}', pending_cell(id))
        return render_cell(c), pend
    return render_cell(c)


In [ ]:
#| export
@rt
def sync_cell(id:int, source:str) -> str:
    "Update a cell's source WITHOUT executing it -- used by Save to flush any editor content that was never explicitly run (Shift+Enter), matching Jupyter's WYSIWYG save behavior."
    c = nb.get(id)
    if c: c.source = source
    return ''

## Run it

In a notebook, start the server and preview inline. Click a cell's header to
select it, then use command-mode keys: `A`/`B` insert above/below, `D D` delete,
`J`/`K` (or arrows) move selection, `M`/`Y`/`R` change type. In the composer,
`Cmd/Ctrl+/` toggles comments on the selected lines and `Cmd/Ctrl+Shift+-` splits
at the caret. On WSL see the lesson's port notes for reaching it from Windows.

In [ ]:
#| eval: false
srv = JupyUvi(app)
p(index())

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()